In [2]:
import pandas as pd
import numpy as np
import geopandas as gpd
import folium
import json
import random
import math
import requests
from shapely.geometry import Point
from shapely.geometry import shape

In [4]:
# ── Load KCPD_Data.csv ────────────────────────────────────────
df_crews = pd.read_csv("KCPD_Data.csv")
df_crews["Crew"] = df_crews["Crew"].str.strip().str.title()
df_crews = df_crews.dropna(subset=["Lat", "Long"])

# Filter to KC bounding box
df_crews = df_crews[
    (df_crews["Lat"]  > 38.85) & (df_crews["Lat"]  < 39.25) &
    (df_crews["Long"] > -94.85) & (df_crews["Long"] < -94.4)
].copy()

def split_lines(val):
    if pd.isna(val):
        return []
    return [x.strip() for x in str(val).replace(",", "\n").split("\n") if x.strip()]

crew_loc = {row["Crew"]: (row["Lat"], row["Long"]) for _, row in df_crews.iterrows()}

catalyst_to_crew = {}
crew_catalysts   = {row["Crew"]: [] for _, row in df_crews.iterrows()}

for _, row in df_crews.iterrows():
    crew = row["Crew"]
    for cat in split_lines(row.get("Catalysts", "")):
        if cat not in catalyst_to_crew:
            catalyst_to_crew[cat] = (crew, row["Lat"], row["Long"])
        if cat not in crew_catalysts[crew]:
            crew_catalysts[crew].append(cat)

# Per-catalyst type, status, assigned worker
catalyst_type_raw     = {}   # raw type string (Stipend / Network / Outreach / Mentor)
catalyst_status_map   = {}
catalyst_assigned_map = {}

for _, row in df_crews.iterrows():
    cats     = split_lines(row.get("Catalysts", ""))
    types    = split_lines(row.get("Catalyst Type", ""))
    statuses = split_lines(row.get("Catalyst Status", ""))
    assigned = split_lines(row.get("Assigned to", ""))
    n = len(cats)
    types    += [""] * max(0, n - len(types))
    statuses += [""] * max(0, n - len(statuses))
    assigned += [""] * max(0, n - len(assigned))
    for cat, ctype, cstatus, cassigned in zip(cats, types, statuses, assigned):
        catalyst_type_raw[cat]     = ctype.strip()
        catalyst_status_map[cat]   = cstatus.strip()
        catalyst_assigned_map[cat] = cassigned.strip()

# Per-catalyst enrollment date
from datetime import datetime
catalyst_date = {}
for _, row in df_crews.iterrows():
    cats  = split_lines(row.get("Catalysts", ""))
    dates = split_lines(row.get("Date of Enrollment", ""))
    for cat, d in zip(cats, dates):
        try:
            catalyst_date[cat] = pd.to_datetime(d.strip()).date()
        except:
            catalyst_date[cat] = None

all_dates = [d for d in catalyst_date.values() if d]
DATE_MIN = min(all_dates).strftime("%Y-%m-%d")
DATE_MAX = max(all_dates).strftime("%Y-%m-%d")

# Crew affiliations & conflicts
crew_affiliations, crew_conflicts = [], []
crew_names_lower = {c.lower(): c for c in crew_loc}

def resolve_crew(name):
    return crew_names_lower.get(name.strip().lower()) if name else None

for _, row in df_crews.iterrows():
    crew = row["Crew"]
    for other in split_lines(row.get("Affiliations", "")):
        r = resolve_crew(other)
        if r and r != crew:
            pair = tuple(sorted([crew, r]))
            if pair not in crew_affiliations:
                crew_affiliations.append(pair)
    for other in split_lines(row.get("Conflicts w Other Crews", "")):
        r = resolve_crew(other)
        if r and r != crew:
            pair = tuple(sorted([crew, r]))
            if pair not in crew_conflicts:
                crew_conflicts.append(pair)

# Catalyst-network orgs
org_to_crews = {}
for _, row in df_crews.iterrows():
    for org in split_lines(row.get("Catalyst Network", "")):
        org_to_crews.setdefault(org, []).append(row["Crew"])

org_loc = {}
for org, crews in org_to_crews.items():
    lats = [crew_loc[c][0] for c in crews if c in crew_loc]
    lons = [crew_loc[c][1] for c in crews if c in crew_loc]
    if lats:
        org_loc[org] = (sum(lats)/len(lats) + 0.012, sum(lons)/len(lons))

crew_org_edges = [
    (row["Crew"], org)
    for _, row in df_crews.iterrows()
    for org in split_lines(row.get("Catalyst Network", ""))
    if org in org_loc
]

# Color palette — one color per catalyst type
TYPE_COLORS = {
    "Stipend":  "#22a861",
    "Network":  "#cc5ca8",
    "Outreach": "#4e8fd4",
    "Mentor":   "#d4920c",
}
DEFAULT_COLOR = "#888888"

def cat_color(cat):
    return TYPE_COLORS.get(catalyst_type_raw.get(cat, ""), DEFAULT_COLOR)

def cat_type_label(cat):
    return catalyst_type_raw.get(cat, "Unknown")

from collections import Counter
print(f"Crews: {len(df_crews)}  |  Catalysts: {len(catalyst_to_crew)}")
print(f"Affiliations: {len(crew_affiliations)}  |  Conflicts: {len(crew_conflicts)}")
print(f"Enrollment date range: {DATE_MIN} -> {DATE_MAX}")
print("Catalyst type breakdown:", Counter(catalyst_type_raw.values()))


Crews: 8  |  Catalysts: 58
Affiliations: 0  |  Conflicts: 13
Enrollment date range: 2019-04-19 -> 2025-10-13
Catalyst type breakdown: Counter({'Outreach': 20, 'Stipend': 15, 'Network': 13, 'Mentor': 10})


In [6]:
# Generate synthetic shooting points — wider spread to color more tracts
random.seed(42)
synthetic_rows = []

for _, row in df_crews.iterrows():
    n = int(pd.to_numeric(row["Conflicts Reported"], errors="coerce") or 0)
    for _ in range(n * 25):  # more points for better tract coverage
        jlat = row["Lat"] + random.gauss(0, 0.04)  # ~4km spread (was 0.018)
        jlon = row["Long"] + random.gauss(0, 0.04)
        synthetic_rows.append({"Lat": jlat, "Long": jlon, "conflicts": 1})

df_synthetic = pd.DataFrame(synthetic_rows)
print(f"Generated {len(df_synthetic)} synthetic incident points")

Generated 1200 synthetic incident points


In [8]:


print("Loading Census tract polygons for Jackson County MO + Wyandotte County KS…")

# Missouri tracts — Jackson County (FIPS 29-095)
gdf_mo = gpd.read_file(
    "https://www2.census.gov/geo/tiger/GENZ2020/shp/cb_2020_29_tract_500k.zip"
)
gdf_mo = gdf_mo[gdf_mo["COUNTYFP"] == "095"].copy()
gdf_mo["nbhd_name"] = "MO Tract " + gdf_mo["TRACTCE"].astype(str)
gdf_mo = gdf_mo.to_crs("EPSG:4326")

# Kansas tracts — Wyandotte County (FIPS 20-209)
gdf_ks = gpd.read_file(
    "https://www2.census.gov/geo/tiger/GENZ2020/shp/cb_2020_20_tract_500k.zip"
)
gdf_ks = gdf_ks[gdf_ks["COUNTYFP"] == "209"].copy()
gdf_ks["nbhd_name"] = "KS Tract " + gdf_ks["TRACTCE"].astype(str)
gdf_ks = gdf_ks.to_crs("EPSG:4326")

# Merge both into one GeoDataFrame
gdf_nbhd = pd.concat([gdf_mo, gdf_ks], ignore_index=True)
gdf_nbhd = gdf_nbhd[["nbhd_name", "geometry"]].copy()
gdf_nbhd = gdf_nbhd.set_geometry("geometry")
print(f"  ✓ {len(gdf_mo)} MO tracts + {len(gdf_ks)} KS tracts = {len(gdf_nbhd)} total")

# Spatial-join synthetic incidents → tracts
gdf_points = gpd.GeoDataFrame(
    df_synthetic,
    geometry=gpd.points_from_xy(df_synthetic["Long"], df_synthetic["Lat"]),
    crs="EPSG:4326",
)

joined = gpd.sjoin(
    gdf_points, gdf_nbhd[["nbhd_name", "geometry"]],
    how="left", predicate="within"
)
nbhd_counts = joined.groupby("nbhd_name")["conflicts"].sum().fillna(0).astype(int)
gdf_nbhd["conflict_count"] = (
    gdf_nbhd["nbhd_name"].map(nbhd_counts).fillna(0).astype(int)
)

print(f"Total tracts: {len(gdf_nbhd)}")
print("Conflict counts (non-zero):")
print(gdf_nbhd[gdf_nbhd["conflict_count"] > 0][["nbhd_name", "conflict_count"]])

Loading Census tract polygons for Jackson County MO + Wyandotte County KS…
  ✓ 227 MO tracts + 64 KS tracts = 291 total
Total tracts: 291
Conflict counts (non-zero):
           nbhd_name  conflict_count
0    MO Tract 005300               5
1    MO Tract 006700               4
2    MO Tract 008200               8
3    MO Tract 010105               4
5    MO Tract 002200              10
..               ...             ...
279  KS Tract 980500               2
282  KS Tract 041200               1
284  KS Tract 045200               8
286  KS Tract 981200               3
288  KS Tract 043905               1

[160 rows x 2 columns]


In [9]:
CREW_ORANGE = "#E8782A"

# ── Base map ──────────────────────────────────────────────────
m = folium.Map(location=[39.06, -94.57], zoom_start=12, tiles="CartoDB positron")

# ── Choropleth ────────────────────────────────────────────────
nbhd_geojson = json.loads(gdf_nbhd.to_json())
for i, feat in enumerate(nbhd_geojson["features"]):
    feat["id"] = gdf_nbhd.iloc[i]["nbhd_name"]
    feat["properties"]["conflict_count"] = int(gdf_nbhd.iloc[i]["conflict_count"])
    feat["properties"]["nbhd_name"]      = gdf_nbhd.iloc[i]["nbhd_name"]

def style_function(feature):
    count = feature["properties"].get("conflict_count", 0)
    if   count == 0:   fill, opacity = "#2d8c4e", 0.35
    elif count == 1:   fill, opacity = "#78b85e", 0.55
    elif count <= 3:   fill, opacity = "#c8d96f", 0.60
    elif count <= 10:  fill, opacity = "#ffe08b", 0.65
    elif count <= 25:  fill, opacity = "#fdbe6f", 0.68
    elif count <= 50:  fill, opacity = "#f4845f", 0.70
    elif count <= 100: fill, opacity = "#e8533e", 0.72
    else:              fill, opacity = "#c0233c", 0.75
    return {"fillColor": fill, "fillOpacity": opacity, "color": "#bbb", "weight": 0.8}

folium.GeoJson(
    nbhd_geojson, style_function=style_function,
    tooltip=folium.GeoJsonTooltip(
        fields=["nbhd_name", "conflict_count"],
        aliases=["Neighbourhood:", "Conflicts reported:"],
        style="font-family:'Helvetica Neue',system-ui,sans-serif;font-size:12px;",
    ),
    name="Conflicts by neighbourhood",
).add_to(m)

# ── Affiliation / conflict / catalyst-net lines ───────────────
affiliations_fg = folium.FeatureGroup(name="Affiliations", show=True)
for a, b in crew_affiliations:
    if a in crew_loc and b in crew_loc:
        folium.PolyLine(locations=[crew_loc[a], crew_loc[b]],
            color="#56B4E9", weight=2.5, opacity=0.7,
            tooltip=f"Affiliation: {a} \u2194 {b}").add_to(affiliations_fg)
affiliations_fg.add_to(m)

conflicts_fg = folium.FeatureGroup(name="Conflicts", show=True)
for a, b in crew_conflicts:
    if a in crew_loc and b in crew_loc:
        folium.PolyLine(locations=[crew_loc[a], crew_loc[b]],
            color="#D55E00", weight=2.5, opacity=0.7, dash_array="10,6",
            tooltip=f"Conflict: {a} \u2194 {b}").add_to(conflicts_fg)
conflicts_fg.add_to(m)

catalyst_net_fg = folium.FeatureGroup(name="Catalyst network links", show=True)
for crew_n, org in crew_org_edges:
    if crew_n in crew_loc and org in org_loc:
        folium.PolyLine(locations=[crew_loc[crew_n], org_loc[org]],
            color="#888", weight=1.2, opacity=0.45, dash_array="2,5",
            tooltip=f"Network: {crew_n} \u2192 {org}").add_to(catalyst_net_fg)
catalyst_net_fg.add_to(m)

# ── Jitter ────────────────────────────────────────────────────
def jitter(lat, lon, seed_key, radius=0.003):
    rnd = random.Random(seed_key)
    angle = rnd.uniform(0, 2 * math.pi)
    r = rnd.uniform(0.3, 1.0) * radius
    return (lat + r * math.cos(angle), lon + r * math.sin(angle))

catalyst_loc = {
    cat: jitter(lat, lon, cat)
    for cat, (crew_n, lat, lon) in catalyst_to_crew.items()
}

TOOLTIP_STYLE = (
    "font-family:'Helvetica Neue',system-ui,sans-serif;"
    "font-size:14px;font-weight:600;"
    "background:rgba(255,255,255,0.97);"
    "border:1px solid #ddd;border-radius:4px;"
    "padding:5px 12px;box-shadow:0 2px 6px rgba(0,0,0,0.12);color:#222;"
)
CAT_TOOLTIP_STYLE = (
    "font-family:'Helvetica Neue',system-ui,sans-serif;"
    "font-size:13px;font-weight:500;"
    "background:rgba(255,255,255,0.97);"
    "border:1px solid #ddd;border-radius:4px;"
    "padding:5px 10px;box-shadow:0 2px 6px rgba(0,0,0,0.12);color:#222;"
)

# ── Crew markers — Boston-style graduated circles ─────────────
crews_fg = folium.FeatureGroup(name="Crews & catalysts", show=True)

for _, row in df_crews.iterrows():
    crew   = row["Crew"]
    cats   = crew_catalysts.get(crew, [])
    n_cats = len(cats)
    radius = (22 if n_cats > 12 else 18 if n_cats >= 9 else
              14 if n_cats >= 6  else 10 if n_cats >= 4 else 6)

    # Group catalysts by type for the popup table
    from collections import defaultdict
    by_type = defaultdict(list)
    for c in cats:
        by_type[catalyst_type_raw.get(c, "Unknown")].append(c)

    def cat_rows(cat_list, color, label):
        if not cat_list:
            return ""
        rows = "".join(
            f'<tr>'
            f'<td style="padding:3px 10px 3px 0;color:#333;font-size:13px;">{c}</td>'
            f'<td style="padding:3px 10px 3px 0;color:#666;font-size:12px;">{catalyst_status_map.get(c,"")}</td>'
            f'<td style="padding:3px 0;color:#888;font-size:12px;">{catalyst_assigned_map.get(c,"")}</td>'
            f'</tr>'
            for c in cat_list
        )
        return (
            f'<tr><td colspan="3" style="padding-top:9px;font-weight:700;'
            f'color:{color};font-size:13px;">{label}</td></tr>' + rows
        )

    popup_html = (
        f'<div style="font-family:\'Helvetica Neue\',system-ui,sans-serif;min-width:300px;max-width:400px;">'
        f'<div style="font-size:16px;font-weight:bold;color:{CREW_ORANGE};margin-bottom:3px;">{crew}</div>'
        f'<div style="font-size:13px;color:#555;margin-bottom:10px;">{n_cats} catalyst(s)</div>'
        f'<table style="font-size:13px;border-collapse:collapse;width:100%;">'
        f'<tr style="border-bottom:1px solid #eee;">'
        f'<th style="text-align:left;font-size:11px;color:#aaa;padding-bottom:4px;">Name</th>'
        f'<th style="text-align:left;font-size:11px;color:#aaa;">Status</th>'
        f'<th style="text-align:left;font-size:11px;color:#aaa;">Assigned to</th>'
        f'</tr>'
        + "".join(
            cat_rows(by_type[t], TYPE_COLORS.get(t, DEFAULT_COLOR), t)
            for t in ["Stipend","Network","Outreach","Mentor","Unknown"]
            if by_type[t]
        )
        + '</table></div>'
    )

    folium.CircleMarker(
        location=[row["Lat"], row["Long"]], radius=radius,
        color="#c45a1a", weight=2, fill=True,
        fill_color=CREW_ORANGE, fill_opacity=0.85,
        popup=folium.Popup(popup_html, max_width=420),
        tooltip=folium.Tooltip(f"{crew} ({n_cats} catalysts)", style=TOOLTIP_STYLE),
    ).add_to(crews_fg)

    folium.Marker(
        location=[row["Lat"], row["Long"]],
        icon=folium.DivIcon(
            html=(
                f'<div style="font-size:13px;color:#333;font-weight:700;'
                f'text-shadow:0 0 4px #fff, 0 0 8px #fff;'
                f'white-space:nowrap;transform:translate(18px,-7px);">{crew}</div>'
            ),
            icon_size=(160, 20), icon_anchor=(0, 0),
        ),
    ).add_to(crews_fg)

# ── Catalyst dots — colored by type ──────────────────────────
for cat, (jlat, jlon) in catalyst_loc.items():
    color  = cat_color(cat)
    label  = cat_type_label(cat)
    crew_n = catalyst_to_crew[cat][0]
    enroll = str(catalyst_date.get(cat)) if catalyst_date.get(cat) else ""
    status = catalyst_status_map.get(cat, "")
    worker = catalyst_assigned_map.get(cat, "")

    type_badge = (
        f'<span style="display:inline-block;padding:1px 7px;border-radius:10px;'
        f'background:{color};color:#fff;font-size:11px;font-weight:600;">{label}</span>'
    )

    def field_row(lbl, val, vc="#333"):
        if not val: return ""
        return (
            f'<tr><td style="padding:3px 10px 3px 0;color:#888;font-size:12px;'
            f'white-space:nowrap;font-weight:500;">{lbl}</td>'
            f'<td style="padding:3px 0;color:{vc};font-size:13px;font-weight:500;">{val}</td></tr>'
        )

    cat_popup_html = (
        f'<div style="font-family:\'Helvetica Neue\',system-ui,sans-serif;'
        f'min-width:200px;max-width:260px;padding:2px;">'
        f'<div style="font-size:15px;font-weight:700;color:#222;margin-bottom:6px;">{cat}</div>'
        f'{type_badge}'
        f'<table style="border-collapse:collapse;width:100%;margin-top:8px;">'
        + field_row("Crew",        crew_n)
        + field_row("Enrolled",    enroll)
        + field_row("Status",      status)
        + field_row("Assigned to", worker)
        + '</table></div>'
    )

    folium.CircleMarker(
        location=[jlat, jlon], radius=5,
        color=color, weight=1.2, fill=True, fill_color=color, fill_opacity=0.85,
        tooltip=folium.Tooltip(
            f"{cat}  \u00b7  {label}  \u00b7  {crew_n}",
            style=CAT_TOOLTIP_STYLE,
        ),
        popup=folium.Popup(cat_popup_html, max_width=280),
    ).add_to(crews_fg)

# ── Org nodes ─────────────────────────────────────────────────
for org, (olat, olon) in org_loc.items():
    n_crews = len(org_to_crews.get(org, []))
    folium.CircleMarker(
        location=[olat, olon], radius=8,
        color="#555", weight=1.5, fill=True, fill_color="#999", fill_opacity=0.6,
        tooltip=folium.Tooltip(f"Org: {org} ({n_crews} crews)", style=CAT_TOOLTIP_STYLE),
    ).add_to(crews_fg)
    folium.Marker(
        location=[olat, olon],
        icon=folium.DivIcon(
            html=(
                f'<div style="font-size:13px;color:#333;font-weight:700;'
                f'text-shadow:0 0 4px #fff, 0 0 8px #fff;'
                f'white-space:nowrap;transform:translate(12px,-6px);">{org}</div>'
            ),
            icon_size=(160, 18), icon_anchor=(0, 0),
        ),
    ).add_to(crews_fg)

crews_fg.add_to(m)
print(f"Markers: {len(df_crews)} crews, {len(catalyst_loc)} catalysts, {len(org_loc)} org nodes")


Markers: 8 crews, 58 catalysts, 4 org nodes


In [10]:
folium.LayerControl(position="topright", collapsed=True).add_to(m)

TOTAL_CATS = len(catalyst_to_crew)
CUR_YEAR   = datetime.now().year
TODAY      = datetime.now().strftime("%Y-%m-%d")

# ── Title bar ─────────────────────────────────────────────────
title_html = f"""
<div id="kc-titlebar" style="
     position:fixed;top:10px;left:50%;transform:translateX(-50%);z-index:9999;
     background:rgba(255,255,255,0.94);padding:8px 20px;border-radius:4px;
     font-family:'Helvetica Neue',system-ui,sans-serif;color:#222;
     font-size:16px;font-weight:600;
     box-shadow:0 1px 4px rgba(0,0,0,0.12);border:1px solid #ddd;white-space:nowrap;">
    <span id="kc-title-range">Kansas City Uncornered &mdash; Crew &amp; Catalyst Network</span>
    <span id="kc-title-n" style="color:#888;font-weight:400;">&nbsp;&middot;&nbsp;N&nbsp;=&nbsp;{TOTAL_CATS:,}</span>
    &nbsp;&amp;&nbsp;
    <span style="color:{CREW_ORANGE};font-weight:600;">Crews</span>
</div>
"""
m.get_root().html.add_child(folium.Element(title_html))

# ── Date-filter widget (bottom-right, Boston style) ───────────
# Catalyst enrollment dates are embedded so filtering is fully client-side.
import json as _json
catalyst_json = _json.dumps([
    {
        "name":   cat,
        "date":   str(catalyst_date.get(cat, "")) if catalyst_date.get(cat) else "",
        "crew":   catalyst_to_crew[cat][0],
        "type":   catalyst_type_raw.get(cat, ""),
    }
    for cat in catalyst_to_crew
])

filter_html = f"""
<div id="kc-filter-panel" style="
     position:fixed;bottom:20px;right:12px;z-index:9999;
     background:rgba(255,255,255,0.96);padding:14px 18px;border-radius:6px;
     font-family:'Helvetica Neue',system-ui,sans-serif;color:#333;
     font-size:13px;line-height:1.9;
     box-shadow:0 2px 8px rgba(0,0,0,0.14);border:1px solid #ddd;min-width:220px;">

  <div style="font-size:14px;font-weight:700;margin-bottom:10px;color:#222;">Enrollment Date Filter</div>

  <label style="display:block;font-size:12px;color:#666;margin-bottom:3px;">Start date</label>
  <input id="kc-start" type="date" value="{DATE_MIN}" min="{DATE_MIN}" max="{TODAY}"
         style="width:100%;padding:5px 8px;border:1px solid #ccc;border-radius:3px;
                font-size:13px;box-sizing:border-box;margin-bottom:10px;">

  <label style="display:block;font-size:12px;color:#666;margin-bottom:3px;">End date</label>
  <input id="kc-end" type="date" value="{DATE_MAX}" min="{DATE_MIN}" max="{TODAY}"
         style="width:100%;padding:5px 8px;border:1px solid #ccc;border-radius:3px;
                font-size:13px;box-sizing:border-box;margin-bottom:12px;">

  <div style="display:flex;gap:8px;">
    <button onclick="applyKCFilter()" style="
        flex:1;padding:7px 0;background:#3a7abf;color:#fff;
        border:none;border-radius:3px;font-size:13px;font-weight:600;cursor:pointer;">Apply</button>
    <button onclick="setKCYTD()" title="Year-to-date: Jan 1 {CUR_YEAR} to today" style="
        flex:1;padding:7px 0;background:#2d8c4e;color:#fff;
        border:none;border-radius:3px;font-size:13px;font-weight:600;cursor:pointer;">YTD</button>
  </div>
  <div style="margin-top:8px;font-size:11px;color:#999;">YTD = Jan 1 {CUR_YEAR} to today</div>
  <div id="kc-filter-count" style="font-size:11px;color:#555;margin-top:4px;"></div>
</div>

<script>
var KC_CATS = {catalyst_json};

// Map catalyst name -> SVG/circle element after map loads
var catElements = {{}};

window.addEventListener('load', function() {{
  setTimeout(function() {{
    // Associate each catalyst circle with its name via tooltip text
    document.querySelectorAll('.leaflet-interactive').forEach(function(el) {{
      var tip = el.getAttribute('title') || '';
      KC_CATS.forEach(function(c) {{
        if (tip.indexOf(c.name + '  \u00b7') === 0) {{
          if (!catElements[c.name]) catElements[c.name] = [];
          catElements[c.name].push(el);
        }}
      }});
    }});
    console.log('[KCMap] Ready. Indexed ' + Object.keys(catElements).length + ' catalysts.');
  }}, 800);
}});

function applyKCFilter() {{
  var startVal = document.getElementById('kc-start').value;
  var endVal   = document.getElementById('kc-end').value;
  if (!startVal || !endVal) {{ alert('Please select both a start and end date.'); return; }}
  var start = new Date(startVal + 'T00:00:00Z');
  var end   = new Date(endVal   + 'T23:59:59Z');
  if (start > end) {{ alert('Start date must be before end date.'); return; }}

  var shown = 0, hidden = 0;
  KC_CATS.forEach(function(c) {{
    var vis = true;
    if (c.date) {{
      var d = new Date(c.date + 'T00:00:00Z');
      vis = (d >= start && d <= end);
    }}
    var els = catElements[c.name] || [];
    els.forEach(function(el) {{
      el.style.display = vis ? '' : 'none';
    }});
    if (vis) shown++; else hidden++;
  }});

  var fmt = function(s) {{
    return new Date(s + 'T00:00:00Z')
      .toLocaleDateString('en-US', {{month:'short', year:'numeric', timeZone:'UTC'}});
  }};
  document.getElementById('kc-title-range').textContent =
    'Kansas City Uncornered \u2014 Enrolled ' + fmt(startVal) + ' to ' + fmt(endVal);
  document.getElementById('kc-title-n').textContent =
    ' \u00b7 N\u00a0=\u00a0' + shown;
  document.getElementById('kc-filter-count').textContent =
    shown + ' catalysts shown, ' + hidden + ' filtered out';
}}

function setKCYTD() {{
  var today = new Date();
  var y  = today.getFullYear();
  var mm = String(today.getMonth() + 1).padStart(2, '0');
  var dd = String(today.getDate()).padStart(2, '0');
  document.getElementById('kc-start').value = y + '-01-01';
  document.getElementById('kc-end').value   = y + '-' + mm + '-' + dd;
  applyKCFilter();
}}
</script>
"""
m.get_root().html.add_child(folium.Element(filter_html))
print("Date-filter widget added")

# ── Legend (Boston style) ─────────────────────────────────────
legend_html = f"""
<div style="position:fixed;bottom:40px;left:20px;z-index:9999;
     background:rgba(255,255,255,0.94);padding:16px 20px;border-radius:4px;
     font-family:'Helvetica Neue',system-ui,sans-serif;color:#333;
     font-size:13px;line-height:1.8;max-width:260px;
     box-shadow:0 1px 4px rgba(0,0,0,0.12);border:1px solid #ddd;">

  <div style="font-size:15px;font-weight:700;margin-bottom:5px;">KC Crews</div>
  <div style="font-size:11px;color:#666;margin-bottom:5px;">Number of Catalysts</div>
  {"".join([
      f'<div style="display:flex;align-items:center;gap:10px;margin-top:4px;">'
      f'<div style="width:{sz}px;height:{sz}px;border-radius:50%;'
      f'background:{CREW_ORANGE};border:2px solid #c45a1a;opacity:0.85;flex-shrink:0;"></div>'
      f'<span>{lbl}</span></div>'
      for sz, lbl in [(28,">12"),(22,"9"),(16,"6"),(12,"4"),(8,"<4")]
  ])}

  <div style="margin-top:12px;padding-top:10px;border-top:1px solid #ddd;
       font-size:15px;font-weight:700;margin-bottom:5px;">Catalyst Type</div>
  {"".join([
      f'<div style="display:flex;align-items:center;gap:10px;margin-top:4px;">'
      f'<div style="width:12px;height:12px;border-radius:50%;background:{col};flex-shrink:0;"></div>'
      f'<span>{lbl}</span></div>'
      for col, lbl in [
          ("#22a861","Stipend"),
          ("#cc5ca8","Network"),
          ("#4e8fd4","Outreach"),
          ("#d4920c","Mentor"),
      ]
  ])}

  <div style="margin-top:12px;padding-top:10px;border-top:1px solid #ddd;
       font-size:15px;font-weight:700;margin-bottom:5px;">Conflicts by Neighbourhood</div>
  <div style="font-size:11px;color:#666;margin-bottom:5px;">Conflicts Reported &middot; aggregated</div>
  {"".join([
      f'<div style="display:flex;align-items:center;gap:8px;margin-top:3px;">'
      f'<div style="width:20px;height:14px;background:{col};border:1px solid #bbb;flex-shrink:0;"></div>'
      f'<span>{lbl}</span></div>'
      for col, lbl in [
          ("#2d8c4e","0"), ("#78b85e","1"), ("#c8d96f","2-3"),
          ("#ffe08b","4-10"), ("#fdbe6f","11-25"),
          ("#f4845f","26-50"), ("#e8533e","51-100"), ("#c0233c","100+"),
      ]
  ])}

  <div style="margin-top:12px;padding-top:10px;border-top:1px solid #ddd;">
    <div style="display:flex;align-items:center;gap:8px;margin-top:4px;">
      <svg width="22" height="4"><line x1="0" y1="2" x2="22" y2="2"
        stroke="#56B4E9" stroke-width="2.5"/></svg><span>Affiliation</span></div>
    <div style="display:flex;align-items:center;gap:8px;margin-top:4px;">
      <svg width="22" height="4"><line x1="0" y1="2" x2="22" y2="2"
        stroke="#D55E00" stroke-width="2.5" stroke-dasharray="4,3"/></svg><span>Conflict</span></div>
    <div style="display:flex;align-items:center;gap:8px;margin-top:4px;">
      <svg width="22" height="4"><line x1="0" y1="2" x2="22" y2="2"
        stroke="#888" stroke-width="1.5" stroke-dasharray="2,4"/></svg><span>Catalyst network link</span></div>
  </div>
</div>
"""
m.get_root().html.add_child(folium.Element(legend_html))
print("Legend added")

m.save("kc_crew_network_map.html")
print("Saved -> kc_crew_network_map.html")
m


Date-filter widget added
Legend added
Saved -> kc_crew_network_map.html
